# 03. Lecke - Ügynöki Tervezési Minták

Ebben a leckében három alapvető tervezési mintát vizsgálunk meg hatékony AI ügynökök építéséhez:

1. **Világos ügynöki utasítások** — Pontos, szerepet meghatározó promptok készítése az ügynök viselkedésének irányítására
2. **Strukturált kimenet Pydantic modellekkel** — Annak biztosítása, hogy az ügynökök kiszámítható, validált adatokat adjanak vissza
3. **Egyfeladatú ügynökök** — Olyan fókuszált ügynökök tervezése, akik mind egy dolgot jól csinálnak

Minden mintát egy **utazási célpont ajánló** példán alkalmazunk, és fokozatosan építünk egy olyan rendszert, amely célpontokat tud javasolni, elérhetőséget ellenőrizni és logisztikát kezelni.


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Minta 1: Világos ügynöki utasítások

A leghatásosabb minta egyben a legegyszerűbb is: világos, részletes utasítások megfogalmazása az ügynököd számára.

A jó utasítások meghatározzák:
- **Ki** az ügynök (személyiség és hangnem)
- **Mit** kell tennie (lépésről lépésre feladatok)
- **Hogyan** kell viselkednie (korlátok és stílus)

Lent egy utazási concierge ügynököt hozunk létre, explicit utasításokkal, amelyek alakítják minden általa adott választ.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Minta 2: Strukturált kimenet Pydantic modellekkel

A szabad formátumú szöveg hasznos a beszélgetéshez, de az alrendszerek strukturált adatokat igényelnek.
A **Pydantic modellek** és egy **eszközfüggvény** párosításával képesek vagyunk:

- Pontos sémát definiálni az ügynök kimenetéhez
- Automatikusan érvényesíteni a válaszokat
- Megbízhatóan integrálni az ügynök eredményeit az alkalmazás logikájába

Az érvényesítés kulcsa a `response_format` átadása az ügynök futtatásakor. Ez arra kényszeríti a
modellt, hogy ellenőrzött `TravelRecommendations` objektumot adjon vissza (elérhető a `response.value`-ban)
szabad formátumú szöveg helyett. A `get_destination_details` eszköz szintén tipizált
`DestinationRecommendation`-t ad vissza, így az adatok az elejétől a végéig strukturáltak maradnak.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Minta 3: Egyetlen felelősségű ügynökök

Összetett feladatok előnyös, ha több, egyetlen felelősségre fókuszált ügynökre bontjuk őket:

- Egy **Úti cél szakértő**, aki ismeri a helyeket és az elérhetőségeket
- Egy **Logisztikai tervező**, aki kezeli a járatokat, hoteleket és a menetrendeket

Ez párhuzamba állítható a szoftvermérnöki *feladatkörök szétválasztása* elvével — így minden ügynök könnyebben tesztelhető, karbantartható és fejleszthető önállóan.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Összefoglaló

Ebben a leckében három ügynöki tervezési mintát alkalmaztunk egy utazási ajánló példában:

| Minta | Fő ötlet | Előny |
|---|---|---|
| **Világos utasítások** | Határozd meg előre a személyiséget, felelősségeket és korlátokat | Következetes, márkának megfelelő ügynöki viselkedés |
| **Strukturált kimenet** | Használj Pydantic modelleket válaszformátumként | Ellenőrzött, gép által olvasható eredmények |
| **Egyetlen felelősség** | Adj minden ügynöknek egy fókuszált feladatot | Könnyebb tesztelni, karbantartani és összerakni |

Ezek a minták természetesen kombinálhatók — világos utasításokat és strukturált kimenetet is alkalmazhatsz egyetlen felelősségű ügynökön belül, hogy robusztus, éles használatra kész rendszereket építs. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
